In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 19:25:03.634886: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 19:25:04.420762: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 19:25:05,698 [DEBUG] [Rain] Rain is initialized
2023-07-04 19:25:05,700 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 19:25:05,700 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 19:25:05,702 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 19:25:05,703 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 19:25:05,704 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 19:25:05,705 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 19:25:05,706 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 19:25:05,711 [DEBUG] [Rain] Creating workers
2023-07-04 19:25:05,726 [INFO] [Provisioner] provisioner is serving
2023-07-04 19:25:05,727 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 19:25:05,729 [INFO] [Coordinator] coordinator is serving
2023-07-04 19:25:05,729 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 19:25:05,735 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 19:25:05,737 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 19:25:05,739 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 19:25:05,740 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 19:25:05,743 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 19:25:05,744 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 19:25:05,747 [INFO] [W

157/157 [==============================] - 3s 10ms/step - loss: 0.7027 - accuracy: 0.7774


2023-07-04 19:25:25,952 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:25,954 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider
157/157 [==============================] - 3s 11ms/step - loss: 0.7038 - accuracy: 0.7779
sending data to divider


2023-07-04 19:25:25,958 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:25,962 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 19:25:25,982 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:25,984 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2


...

2023-07-04 19:25:36,514 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:36,516 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-04 19:25:36,531 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:36,533 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-04 19:25:36,776 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 19:25:36,789 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-04 19:25:36,825 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 19:25:36,829 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
2023-07-04 19:25:36,831 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 s

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.1471 - accuracy: 0.9551

Test accuracy: 95.5%


In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.1471 - accuracy: 0.9551

Test accuracy: 95.5%


In [11]:
# model = rain.train(X_train, y_train, strategy='sync')

In [12]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 19:25:37,805 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 19:25:37,808 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 19:25:37,809 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 19:25:37,811 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 19:25:37,812 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 19:25:37,814 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 19:25:37,815 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 19:25:37,

157/157 [==============================] - 3s 7ms/step - loss: 0.2056 - accuracy: 0.9385


2023-07-04 19:25:57,593 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to dividersending data to divider

sending data to divider


2023-07-04 19:25:57,593 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:57,596 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:57,597 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:57,598 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:25:57,598 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider beg

...

2023-07-04 19:26:05,812 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:26:05,816 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_3_trained.pkl from worker1


sending data to divider


DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_3_trained.pkl from worker1


157/157 [==============================] - 4s 10ms/step - loss: 0.1593 - accuracy: 0.9517
sending data to divider


2023-07-04 19:26:05,954 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 19:26:05,956 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/2_3_trained.pkl from worker2
2023-07-04 19:26:06,025 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-04 19:26:06,026 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 19:26:06,085 [DEBUG] [DividerAm

In [13]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))